In [2]:
!pip install tensorflow scikit-learn

In [17]:
import pandas as pd

df= pd.read_csv("IMDB Dataset.csv")
df["label"]= df["sentiment"].map({"negative":0,"positive":1})
#cleaning HTML line-breaking tags
df["review"]=df["review"].str.replace(r"<br\s*/?"," ", regex=True)
#remove non-ASCII characters (drops the /x96 char that broke .keras saving)
df["review"]=df["review"].str.encode("ascii","ignore").str.decode("ascii")

print(df.shape)
df.head()

(50000, 3)


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. > >The filming...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [19]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(
    df["review"].values, df["label"].values,
    test_size=0.2, random_state=42
)
print("Train:",len(x_train),"Test:",len(x_test))

Train: 40000 Test: 10000


In [20]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

max_tokens=10000    #keep the 10000 most common words
max_len=200         #cap each review at 200 words

vectorize_layer = TextVectorization(       
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_len
)
vectorize_layer.adapt(x_train)
print("Vocabulary learned with default cleaning!")

Vocabulary learned with default cleaning!


In [21]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding,GlobalAveragePooling1D,Dense

model= Sequential([
    vectorize_layer,
    Embedding(max_tokens,16),
    GlobalAveragePooling1D(),
    Dense(16,activation="relu"),
    Dense(1,activation="sigmoid")
])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_4            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(optimizer="adam",
             loss="binary_crossentropy",
             metrics=["accuracy"])

history=model.fit(x_train,y_train,
                  epochs=5,
                 validation_split=0.2,
                 batch_size=32)

Epoch 1/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.7533 - loss: 0.5076 - val_accuracy: 0.8521 - val_loss: 0.3520
Epoch 2/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8706 - loss: 0.3083 - val_accuracy: 0.8683 - val_loss: 0.3151
Epoch 3/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8980 - loss: 0.2552 - val_accuracy: 0.8730 - val_loss: 0.3055
Epoch 4/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9109 - loss: 0.2277 - val_accuracy: 0.8737 - val_loss: 0.3035
Epoch 5/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9174 - loss: 0.2094 - val_accuracy: 0.8727 - val_loss: 0.3136


In [23]:
loss,acc=model.evaluate(x_test,y_test)
print(f"Test accuracy: {acc:.3f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8766 - loss: 0.3054
Test accuracy: 0.877


In [24]:
my_reviews=[
    "This movie was absolutely peak,I enjoyed every moment",
    "Complete waste of time,terrible acting and total disaster plot"
]
preds=model.predict(tf.constant(my_reviews))
for reviews,p in zip(my_reviews, preds):
    print(f"{'positive' if p[0]>0.5 else 'negative'}({p[0]:.2f})-{reviews}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
positive(0.83)-This movie was absolutely peak,I enjoyed every moment
negative(0.11)-Complete waste of time,terrible acting and total disaster plot


In [25]:
my_reviews = [
    "An absolutely wonderful film, I loved every minute of it",
    "This was a fantastic movie with brilliant acting",
    "A boring, terrible film that wasted my time",
    "This movie was absolutely peak, I enjoyed every moment"
]
preds = model.predict(tf.constant(my_reviews))
for review, p in zip(my_reviews, preds):
    print(f"{'positive' if p[0] > 0.5 else 'negative'} ({p[0]:.2f}) — {review}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
positive (0.91) — An absolutely wonderful film, I loved every minute of it
positive (0.92) — This was a fantastic movie with brilliant acting
negative (0.10) — A boring, terrible film that wasted my time
positive (0.86) — This movie was absolutely peak, I enjoyed every moment


In [26]:
model.save("imdb_sentiment_model.keras")
print("Model saved as .keras!")

Model saved as .keras!


In [27]:
import os
print(os.path.exists("imdb_sentiment_model.keras"))
size_mb = os.path.getsize("imdb_sentiment_model.keras") / 1_000_000
print(f"File size: {size_mb:.1f} MB")

True
File size: 2.1 MB


In [28]:
import tensorflow as tf
from tensorflow import keras

loaded_model = keras.models.load_model("imdb_sentiment_model.keras")

test = ["An absolutely wonderful film, I loved it",
        "A boring terrible waste of time"]
preds = loaded_model.predict(tf.constant(test))
for review, p in zip(test, preds):
    print(f"{'positive' if p[0] > 0.5 else 'negative'} ({p[0]:.2f}) — {review}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
positive (0.93) — An absolutely wonderful film, I loved it
negative (0.04) — A boring terrible waste of time
